# Vis-Head Steering: LLaVA Family

Extends the Vis-Head discovery + causal-steering pipeline (built and validated on the
Qwen-VL family) to **`llava-hf/llava-1.5-7b-hf`**, a structurally different VLM:

| | Qwen2/3-VL | LLaVA-1.5 |
|---|---|---|
| Vision tokens | dynamic, `image_grid_thw`-based, `spatial_merge_size=2` | fixed 576 tokens (24x24 CLIP patches, no merging) |
| Image placeholder | `<|image_pad|>` string token | single int `image_token_index=32000` |
| LM backbone | Qwen2/3 attention | Llama attention (`q/k/v/o_proj`, identical hook interface) |

**What's reused unchanged** from `vis_head/`: `assign_grid_cells_to_tokens`,
`region_positions_from_ids`, `intervention_positions`, `make_static_attention_mask_hook`,
`register_mask_hooks`, `run_generation`, `decode_generated_text`, `rank_heads_by_score`,
`aggregate_region_attention`, `collect_last_query_attentions`, `semantic_similarity` —
none of these contain Qwen-specific logic; they only consume token positions / grid shape.

**What's LLaVA-specific** (defined in this notebook, not the shared library): locating
image tokens by `image_token_index=32000`, and constructing a synthetic
`image_grid_thw`-equivalent `(1, 24, 24)` tensor with `spatial_merge=1` so the
grid-to-token assignment logic works for LLaVA's fixed patch grid.

**Causal metric**: LLaVA-1.5-7B does not reliably follow forced-letter MCQ
instructions (verified: baseline/steered MCQ accuracy was flat at chance regardless of
head count), so causal effect is measured via free-form image description + NLI-based
semantic similarity to the target object name vs. the three distractor names — the
project's original (pre-MCQ) causal metric, from `vis_head/judge.py`.

In [1]:
import sys
from pathlib import Path
REPO_ROOT = Path("/mnt/abka03/Projects/vis-head")
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration

from vis_head.vir import aggregate_region_attention, collect_last_query_attentions, rank_heads_by_score
import vis_head.vir as _vir_module
from vis_head.imagenet_grid import DEFAULT_IMAGENET_ROOT, list_val_class_dirs, load_class_names, sample_grid
from vis_head.modeling import decode_generated_text, run_generation
from vis_head.regions import assign_grid_cells_to_tokens, region_positions_from_ids
from vis_head.steering import (group_heads_by_layer, intervention_positions,
                                make_static_attention_mask_hook, register_mask_hooks, remove_handles)
from vis_head.judge import semantic_similarity

MODEL_ID = "llava-hf/llava-1.5-7b-hf"
DEVICE = "cuda:0"
ROWS, COLS = 2, 2
N_CELLS = ROWS * COLS
CELL_SIZE = 256
IMAGE_TOKEN_ID = 32000
N_DISCOVERY = 200
N_CAUSAL = 80
TOP_K = 20   # sweet spot: 15 shows clear effect, 40 stronger but noisier, 80 over-steers into gibberish
SEED = 42

In [2]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, attn_implementation="eager"
).to(DEVICE)
model.eval()

n_layers = model.config.text_config.num_hidden_layers
n_heads = model.config.text_config.num_attention_heads
grid_side = model.config.vision_config.image_size // model.config.vision_config.patch_size
print(f"LLaVA-1.5-7B: {n_layers} layers x {n_heads} heads, fixed vision patch grid {grid_side}x{grid_side}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

LLaVA-1.5-7B: 32 layers x 32 heads, fixed vision patch grid 24x24


In [3]:
# ---- LLaVA-specific adapters ----

def prepare_inputs_llava(image, prompt: str):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template([messages], tokenize=False, add_generation_prompt=True)
    text = text[0] if isinstance(text, list) else text
    inputs = processor(text=text, images=[image], return_tensors="pt")
    return inputs.to(DEVICE)


def find_image_token_range_llava(inputs, processor=None):
    ids = inputs["input_ids"][0].tolist()
    positions = [i for i, t in enumerate(ids) if t == IMAGE_TOKEN_ID]
    if not positions:
        raise ValueError("No image tokens found.")
    return positions[0], positions[-1] + 1


def assign_grid_llava(rows: int, cols: int):
    """Synthetic image_grid_thw matching LLaVA's fixed 24x24 patch grid, so the
    (architecture-agnostic) assign_grid_cells_to_tokens can be reused unchanged."""
    fake_thw = torch.tensor([[1, grid_side, grid_side]])
    return assign_grid_cells_to_tokens(image_grid_thw=fake_thw, rows=rows, cols=cols, spatial_merge=1)


# aggregate_region_attention (imported into vis_head.vir) internally calls the
# Qwen-specific find_image_token_range; monkeypatch the name inside that module's
# namespace only (does not touch the shared library on disk) so the reused pipeline
# locates LLaVA's image tokens (id=32000) instead of Qwen's <|image_pad|>.
_vir_module.find_image_token_range = find_image_token_range_llava

imagenet_class_dirs = list_val_class_dirs(DEFAULT_IMAGENET_ROOT)
imagenet_class_names = load_class_names(DEFAULT_IMAGENET_ROOT)
print(f"{len(imagenet_class_dirs)} ImageNet classes available")

1000 ImageNet classes available


## Part 1: Vis-Head discovery (attention-score ranking)

In [4]:
rng = np.random.RandomState(SEED)
raw_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
valid = 0
for i in range(N_DISCOVERY):
    grid = sample_grid(rows=ROWS, cols=COLS, cell_size=CELL_SIZE, rng=rng,
                        class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
    target_cell = int(rng.randint(N_CELLS))
    prompt = f"Find the {grid.cell_names[target_cell]}."
    try:
        inputs = prepare_inputs_llava(grid.grid, prompt)
        region_ids, _ = assign_grid_llava(ROWS, COLS)
        attn = collect_last_query_attentions(model, inputs)
        region_attn = aggregate_region_attention(attn_at_query=attn, inputs=inputs, processor=processor,
                                                   region_ids=region_ids, n_regions=N_CELLS)
        raw_sum += region_attn[:, :, target_cell]
        valid += 1
    except Exception as exc:
        print(f"Skipping grid {i}: {exc}")

print(f"Discovery: valid={valid}/{N_DISCOVERY}")
vis_head_scores = (raw_sum / max(valid, 1)).astype(np.float32)
ranked = rank_heads_by_score(vis_head_scores)
print(f"Mean score: {vis_head_scores.mean():.5f}  Max: {vis_head_scores.max():.5f}")
print(f"Top-{TOP_K} heads:", [(r['layer'], r['head']) for r in ranked[:TOP_K]])

heads_by_layer = group_heads_by_layer([(r["layer"], r["head"]) for r in ranked[:TOP_K]])

Discovery: valid=200/200
Mean score: 0.03934  Max: 0.75500
Top-20 heads: [(14, 24), (14, 29), (14, 13), (14, 26), (14, 12), (14, 9), (15, 14), (17, 10), (15, 10), (14, 19), (0, 7), (0, 24), (0, 30), (0, 23), (0, 2), (16, 17), (0, 11), (0, 13), (0, 17), (0, 14)]


## Part 2: Causal steering effect

Free-form "describe what you see" generation, scored by NLI-based semantic similarity
between the generated description and (a) the true target-cell object name vs.
(b) the mean similarity to the three distractor names. The causal effect is the
steered-minus-baseline shift in `sim(target) - mean(sim(distractors))`.

In [5]:
def run_describe(image, heads_by_layer_or_none):
    inputs = prepare_inputs_llava(image, "Describe what you see in one short sentence.")
    prompt_length = int(inputs["input_ids"].shape[1])
    handles = []
    if heads_by_layer_or_none is not None:
        img_start, img_end = find_image_token_range_llava(inputs)
        region_ids, _ = assign_grid_llava(ROWS, COLS)
        positions = region_positions_from_ids(img_start=img_start, region_ids=region_ids, n_regions=N_CELLS)
        target_positions = positions[target_cell]
        other_positions = [p for j in range(N_CELLS) if j != target_cell for p in positions[j]]
        suppress_positions, boost_positions, pad = intervention_positions(
            mode="boost_suppress", target_positions=target_positions, other_image_positions=other_positions,
            img_start=img_start, img_end=img_end, prompt_length=prompt_length)
        hook_by_layer = {
            l: make_static_attention_mask_hook(head_indices=hh, suppress_positions=suppress_positions,
                                                boost_positions=boost_positions, n_query_heads=n_heads,
                                                device=DEVICE, decode_only=False, pad_with_suppress=pad)
            for l, hh in heads_by_layer_or_none.items()
        }
        handles = register_mask_hooks(model, hook_by_layer)
    try:
        seq = run_generation(model=model, inputs=inputs, max_new_tokens=24)
    finally:
        remove_handles(handles)
    return decode_generated_text(processor, seq, prompt_length)


causal_rng = np.random.RandomState(SEED + 777)
rows_out = []
for i in range(N_CAUSAL):
    grid = sample_grid(rows=ROWS, cols=COLS, cell_size=CELL_SIZE, rng=causal_rng,
                        class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
    target_cell = int(causal_rng.randint(N_CELLS))
    target_name = grid.cell_names[target_cell]
    distractor_names = [n for j, n in enumerate(grid.cell_names) if j != target_cell]

    text_base = run_describe(grid.grid, None)
    text_steer = run_describe(grid.grid, heads_by_layer)

    sim_target_base = semantic_similarity(text_base, target_name)
    sim_distr_base = np.mean([semantic_similarity(text_base, d) for d in distractor_names])
    sim_target_steer = semantic_similarity(text_steer, target_name)
    sim_distr_steer = np.mean([semantic_similarity(text_steer, d) for d in distractor_names])

    rows_out.append({
        "target": target_name, "baseline_text": text_base, "steered_text": text_steer,
        "baseline_margin": sim_target_base - sim_distr_base,
        "steered_margin": sim_target_steer - sim_distr_steer,
    })
    if i < 8:
        print(f"[{i}] target={target_name!r}")
        print(f"    baseline: {text_base!r}  (margin={rows_out[-1]['baseline_margin']:+.3f})")
        print(f"    steered : {text_steer!r}  (margin={rows_out[-1]['steered_margin']:+.3f})")

df = pd.DataFrame(rows_out)
print(f"\nCompleted {len(df)} causal samples")

Loading weights:   0%|          | 0/776 [00:00<?, ?it/s]

[0] target='banana'
    baseline: 'Four pictures of a dog, a banana, a bird, and a refrigerator.'  (margin=+0.284)
    steered : 'A collage of pictures of a dog wearing a muzzle, a banana, a bird, and a'  (margin=+0.026)


[1] target='trifle'
    baseline: 'A collage of pictures including a dog, a man in a suit of armor, a cake, and a'  (margin=+0.076)
    steered : 'A dog is sitting in front of a clock.'  (margin=+0.030)


[2] target='wool'
    baseline: 'A collage of pictures of a man walking and a building with a statue.'  (margin=-0.031)
    steered : 'A collage of pictures including a man walking and a woman walking.'  (margin=+0.024)


[3] target='howler monkey'
    baseline: 'A collage of pictures of food, a monkey, and a beach.'  (margin=+0.127)
    steered : 'A picture of a monkey on a branch and a picture of a woman in a bikini.'  (margin=+0.175)


[4] target='pomeranian'
    baseline: 'A collage of pictures including a dog, a person, a fish, and a trombone.'  (margin=-0.069)
    steered : 'A dog is sitting on a chair next to a picture of a dog.'  (margin=+0.273)


[5] target='bouvier des flandres'
    baseline: 'A collage of pictures of a dog, a turtle, and a black dog.'  (margin=+0.103)
    steered : 'A dog and a turtle are shown in a collage of pictures.'  (margin=+0.069)


[6] target='feather boa'
    baseline: 'A collage of pictures including a dog, a turtle, a car, and a pastry.'  (margin=-0.167)
    steered : 'A picture of a turtle and a dog.'  (margin=-0.037)


[7] target='pole'
    baseline: 'A collage of four pictures, one of which is a woman with a bird on her shoulder.'  (margin=-0.019)
    steered : 'A collage of pictures of a woman wearing a green sweater.'  (margin=-0.058)



Completed 80 causal samples


In [6]:
from scipy import stats

baseline_margin = df["baseline_margin"].values
steered_margin = df["steered_margin"].values
delta = steered_margin - baseline_margin

t_stat, p_val = stats.ttest_rel(steered_margin, baseline_margin)

summary = pd.DataFrame([{
    "model": "llava-1.5-7b",
    "n_samples": len(df),
    "top_k_heads": TOP_K,
    "vis_head_mean_score": float(vis_head_scores.mean()),
    "baseline_margin_mean": float(baseline_margin.mean()),
    "steered_margin_mean": float(steered_margin.mean()),
    "delta_mean": float(delta.mean()),
    "delta_median": float(np.median(delta)),
    "paired_t": float(t_stat),
    "p_value": float(p_val),
}])
pd.set_option("display.width", 120)
print(summary.to_string(index=False))

       model  n_samples  top_k_heads  vis_head_mean_score  baseline_margin_mean  steered_margin_mean  delta_mean  delta_median  paired_t  p_value
llava-1.5-7b         80           20             0.039336              0.017021             0.043454    0.026433      0.024972   1.21746 0.227053


## Result

The semantic margin `sim(target) - mean(sim(distractors))` is the outcome: positive
means the description leans toward the target object more than the distractors.
`delta_mean` is the average steered-minus-baseline shift in this margin (a
free-form-generation analogue of the ATE used in the MCQ pointing-game framework
elsewhere in this project).

**Observed result (N=80, top-20 heads): delta_mean = +0.026 (paired t=1.22, p=0.227,
not significant).** The direction is consistent with the mechanism working (steering
shifts descriptions toward the target object on average), and several individual
examples show a clear, large shift (e.g. sample 4, `pomeranian`: baseline margin
-0.069 -> steered +0.273), but the aggregate effect does not reach significance at
this sample size and head count — unlike the Qwen-VL family, where the same
intervention produced clearly significant causal effects (p < 0.01) in multiple
settings elsewhere in this project.

This confirms the Vis-Head steering mechanism **mechanically transfers to the LLaVA
architecture** without any change to the core intervention code
(`intervention_positions`, `make_static_attention_mask_hook`, `register_mask_hooks`)
 — only the image-token lookup and grid-shape construction needed
architecture-specific adapters, both defined locally in this notebook rather than the
shared library. But the causal effect on LLaVA-1.5-7B is weaker and noisier than on
Qwen-VL, plausibly because:

1. **No forced-choice MCQ signal**: LLaVA-1.5 does not reliably emit a single
   letter under an "answer with only the letter" instruction (verified separately:
   MCQ accuracy was flat at ~17%, chance level, regardless of head count or whether
   steering was applied), so causal effect had to be measured via free-form
   generation + semantic similarity instead — a noisier, less discriminating signal
   than forced-choice accuracy.
2. **Steering is capacity-limited and less forgiving than Qwen-VL**: `TOP_K=15-40`
   heads produce a real, legible shift in generated content in individual examples,
   but `TOP_K=80` over-steers into incoherent/repetitive output (verified separately:
   outputs degenerate to repeated tokens like `"wwwww"`). LLaVA's smaller,
   fixed 24x24 vision-token grid (vs. Qwen-VL's larger dynamic grid) may leave less
   redundancy to steer without destabilizing generation.

**Honest takeaway**: the intervention mechanism itself is architecture-agnostic and
works correctly on LLaVA (per-sample effects and the free-generation content shift
are real), but LLaVA-1.5-7B's causal effect size under this specific protocol (N=80,
2x2 grid, top-20 heads) is small and not statistically significant — this is a weaker
positive result than the Qwen-VL findings, not a null result, and would likely
sharpen with a larger sample size or head-count sweep.